# Problem Book P1, Chapter 8: Inequalities That Count (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import math, random, itertools
from math import comb, gcd, log, log2, sqrt, pi, factorial
from fractions import Fraction
import numpy as np
from sympy import primerange, factorint, divisor_count
import matplotlib.pyplot as plt
random.seed(1); rng = np.random.default_rng(1)

def dists(P):                       # distinct squared distances of a list of points (exact for integer or rounded input)
    return {round((p[0]-q[0])**2 + (p[1]-q[1])**2, 9) for p, q in itertools.combinations(P, 2)}
def adj_random(n, p):
    A = np.triu((rng.random((n, n)) < p).astype(int), 1); return A + A.T
def triangles(A): return int(round(np.trace(A @ A @ A))) // 6

## Part A: checks

**Problem 8.1 (few distances).** The think box: seven points. The regular heptagon gives $3$ distances, the hexagon with its
centre also $3$; and Erdős's bound $\sqrt{(n-2)/2}$ is far below. Part (b): the bound on unit distances, tested on lattice pieces.

In [ ]:
hept = [(math.cos(2*pi*k/7), math.sin(2*pi*k/7)) for k in range(7)]
hexc = [(0, 0)] + [(math.cos(pi*k/3), math.sin(pi*k/3)) for k in range(6)]
print(len(dists(hept)), len(dists(hexc)), sqrt(5/2))
def unit_pairs(P): return sum(1 for p, q in itertools.combinations(P, 2) if abs((p[0]-q[0])**2 + (p[1]-q[1])**2 - 1) < 1e-9)
for k in (3, 5, 8, 12):
    tri = [(i + j/2, j*sqrt(3)/2) for i in range(k) for j in range(k)]
    sq = [(i, j) for i in range(k) for j in range(k)]
    n = k*k; bound = n/4*(1 + sqrt(8*n - 7))
    print(n, unit_pairs(sq), unit_pairs(tri), round(bound, 1))

**Problem 8.2 (incidences by Cauchy–Schwarz).** All lines through at least two points of a $k\times k^2$ grid; the vertical
lines are the only rich ones.

In [ ]:
def grid_lines(P):
    S = set(P); L = {}
    for p, q in itertools.combinations(P, 2):
        a, b = q[1]-p[1], p[0]-q[0]; g = gcd(a, b); a, b = a//g, b//g
        if a < 0 or (a == 0 and b < 0): a, b = -a, -b
        L.setdefault((a, b, a*p[0] + b*p[1]), set()).update([p, q])
    return L
for k in (3, 4, 5, 6):
    P = [(i, j) for i in range(k) for j in range(k*k)]; L = grid_lines(P)
    n, m = len(P), len(L); I = sum(len(s) for s in L.values())
    big = sum(1 for s in L.values() if len(s) >= 2*sqrt(n))
    print(k, n, m, I, round(n*sqrt(m) + m), round(m*sqrt(n) + n), " lines with >= 2 sqrt(n) points:", big, "<=", 2*n/(2*sqrt(n)))

**Problem 8.3 (Szemerédi–Trotter).** The extremal example $P=[k]\times[2k^2]$, lines $y=ax+b$ with $a\in[k]$, $b\in[k^2]$:
$I=k^4$ against $4(nm)^{2/3}+m+4n$. The ratio stays bounded: the exponent $\frac23$ cannot be improved.

In [ ]:
for k in (2, 4, 8, 16, 32):
    n, m = 2*k**3, k**3
    I = sum(1 for a in range(1, k+1) for b in range(1, k*k+1) for x in range(1, k+1) if 1 <= a*x + b <= 2*k*k)
    print(k, I == k**4, round(I / (nm := n*m)**(2/3), 3), I <= 4*nm**(2/3) + m + 4*n)

**Problem 8.4 (Elekes).** $|A+A|\,|A\cdot A|/N^{5/2}$ for intervals, geometric progressions and random sets.

In [ ]:
def sp(A): return len({a+b for a in A for b in A}), len({a*b for a in A for b in A})
for N in (10, 30, 100, 300):
    out = []
    for A in (range(1, N+1), [2**i for i in range(N)], random.sample(range(1, 10*N), N)):
        s, p = sp(A); out.append(round(s*p / N**2.5, 2))
    print(N, out)

**Problem 8.5 (energy, multiplication table).** $E_\times([N])$ by brute force and by the coprime-pairs formula, the ratio
$E_\times/(N^2\log N)$ creeping up towards $12/\pi^2=1.2158$, and the lower bound $N^4/E_\times$ for the table.

In [ ]:
from collections import Counter
def Ex_brute(N):
    c = Counter(a*b for a in range(1, N+1) for b in range(1, N+1)); return sum(v*v for v in c.values())
def Ex_formula(N):
    return sum((N // max(u, v))**2 for u in range(1, N+1) for v in range(1, N+1) if gcd(u, v) == 1)
print(all(Ex_brute(N) == Ex_formula(N) for N in range(1, 40)))
phi = list(range(5001))
for i in range(2, 5001):
    if phi[i] == i:
        for j in range(i, 5001, i): phi[j] -= phi[j] // i
def Ex_fast(N):                        # group the pairs by M = max(u, v)
    return 1 + sum(2*phi[M]*(N // M)**2 for M in range(2, N+1)) if N else 0
for N in (100, 1000, 5000):
    E = Ex_fast(N); distinct = len({a*b for a in range(1, N+1) for b in range(a, N+1)}) if N <= 1000 else None
    print(N, round(E / (N*N*log(N)), 4), round(N**4 / E), distinct)

**Problem 8.6 (triangles from eigenvalues).** Power sums of eigenvalues, and $t/m^{3/2}$ for $K_n$ against $\sqrt2/3=0.4714$.

In [ ]:
for n, p in ((12, 0.3), (30, 0.5), (60, 0.2)):
    A = adj_random(n, p); lam = np.linalg.eigvalsh(A); m = A.sum() // 2; t = triangles(A)
    print(round(sum(lam**2)) == 2*m, round(sum(lam**3)) == 6*t, t <= sqrt(2)/3 * m**1.5)
print([round(comb(n, 3) / comb(n, 2)**1.5, 4) for n in (5, 50, 500, 5000)], round(sqrt(2)/3, 4))

**Problem 8.7 (Goodman).** The bound for $n=6,\dots,10$, and an exhaustive check for $n=6$ over all $2^{15}$ colourings.

In [ ]:
print([math.ceil(comb(n, 3) - Fraction(n, 2) * ((n-1)**2 // 4)) for n in range(6, 11)])
E = list(itertools.combinations(range(6), 2)); idx = {e: i for i, e in enumerate(E)}
T = [(idx[(a, b)], idx[(a, c)], idx[(b, c)]) for a, b, c in itertools.combinations(range(6), 3)]
best = min(sum(1 for x, y, z in T if ((c >> x) & 1) == ((c >> y) & 1) == ((c >> z) & 1)) for c in range(1 << 15))
print("minimum over all colourings of K6:", best)

**Problem 8.8 (walks).** Walk counts against $n\bar d^k$, and $\operatorname{tr}A^4$ for Paley graphs ($1482$ for $p=13$).

In [ ]:
A = adj_random(40, 0.15); n = len(A); d = A.sum() / n; one = np.ones(n)
print([bool(one @ np.linalg.matrix_power(A, k) @ one >= n * d**k) for k in (2, 4, 6)])
for p in (13, 29, 101, 401):
    sq = {x*x % p for x in range(1, p)}
    P = np.array([[1 if (x - y) % p in sq else 0 for y in range(p)] for x in range(p)])
    t4 = int(np.trace(np.linalg.matrix_power(P.astype(np.int64), 4))); d = (p - 1) // 2
    formula = p*d*d + p*d*((p-5)//4)**2 + p*d*((p-1)//4)**2
    print(p, t4, t4 == formula, round(t4 / d**4, 4))

**Problem 8.9 (spanning trees and AM–GM).** Kirchhoff's count against $\frac1n\bigl(\frac{2m}{n-1}\bigr)^{n-1}$; equality for $K_n$ (Cayley).

In [ ]:
def tau(A):
    L = np.diag(A.sum(1)) - A; return round(np.linalg.det(L[1:, 1:].astype(float)))
for n, p in ((8, 0.5), (10, 0.7), (12, 0.9)):
    A = adj_random(n, p); m = A.sum() // 2
    print(n, m, tau(A), round((2*m/(n-1))**(n-1) / n))
for n in (4, 6, 9):
    K = np.ones((n, n), dtype=int) - np.eye(n, dtype=int); print(tau(K), n**(n-2))

**Problem 8.10 (Bollobás).** The extremal system ($a$-sets of $[a+b]$ with their complements) gives equality; a random
greedy system satisfies the inequality; $K_{t+2}$ is $\tau$-critical with $\binom{t+2}2$ edges.

In [ ]:
a, b = 3, 4
sys_ = [(set(S), set(range(a+b)) - set(S)) for S in itertools.combinations(range(a+b), a)]
print(sum(Fraction(1, comb(len(A_) + len(B_), len(A_))) for A_, B_ in sys_))
U = range(9); pairs = []
for _ in range(4000):
    A_ = set(random.sample(U, random.randint(1, 4))); B_ = set(random.sample([x for x in U if x not in A_], random.randint(1, 4)))
    if all(A_ & B2 and A2 & B_ for A2, B2 in pairs): pairs.append((A_, B_))
print(len(pairs), float(sum(Fraction(1, comb(len(x)+len(y), len(x))) for x, y in pairs)))
def cover_number(n, E):
    return min(k for k in range(n+1) for S in itertools.combinations(range(n), k) if all(u in S or v in S for u, v in E))
for t in (1, 2, 3):
    n = t + 2; E = list(itertools.combinations(range(n), 2)); c = cover_number(n, E)
    print(t, c == t + 1, all(cover_number(n, E[:i] + E[i+1:]) == t for i in range(len(E))), len(E) == comb(t+2, 2))

**Problem 8.11 (Littlewood–Offord).** The largest mass of an open interval of length $2$, over all sign patterns, for random
$|a_i|\ge1$ and for all $a_i=1$; and $\binom n{\lfloor n/2\rfloor}2^{-n}\sqrt{\pi n/2}\to1$.

In [ ]:
def max_mass(a):
    s = sorted(sum(e*x for e, x in zip(eps, a)) for eps in itertools.product((1, -1), repeat=len(a)))
    best, j = 0, 0
    for i in range(len(s)):
        while s[i] - s[j] >= 2: j += 1
        best = max(best, i - j + 1)
    return best / 2**len(a)
n = 12
for a in ([1]*n, [1 + random.random() for _ in range(n)], [random.choice((1, 1.5, 3)) for _ in range(n)]):
    print(round(max_mass(a), 5), round(comb(n, n//2) / 2**n, 5))
print([round(comb(n, n//2) / 2**n * sqrt(pi*n/2), 4) for n in (10, 100, 1000, 10000)])

**Problem 8.12 (Harris–Kleitman).** Random increasing families on $[6]$, and the union of the $k$ stars, which reaches $2^n-2^{n-k}$.

In [ ]:
n = 6; full = range(1 << n)
def up(gens): return {s for s in full if any(s & g == g for g in gens)}
ok = True
for _ in range(300):
    A_ = up(random.sample(full, 3)); B_ = up(random.sample(full, 3))
    ok &= (1 << n) * len(A_ & B_) >= len(A_) * len(B_)
print(ok)
for k in range(1, n+1):
    U_ = {s for s in full if s & ((1 << k) - 1)}; print(k, len(U_), 2**n - 2**(n-k), end="; ")

**Problem 8.13 (Sauer–Shelah).** Random families on $[7]$ with their VC-dimension, and intervals on a line.

In [ ]:
def vc(F, n):
    d = 0
    for k in range(1, n+1):
        if any(len({f & sum(1 << i for i in T) for f in F}) == 2**k for T in itertools.combinations(range(n), k)): d = k
        else: break
    return d
n = 7
for size in (10, 30, 60, 100):
    F = set(random.sample(range(1 << n), size)); d = vc(F, n); print(size, d, size <= sum(comb(n, i) for i in range(d+1)))
I = {0} | {sum(1 << i for i in range(l, r)) for l in range(n) for r in range(l+1, n+1)}
print(len(I), 1 + n + comb(n, 2), vc(I, n))

**Problem 8.14 (edges inside a set of vertices of the cube).** Exhaustive over all $2^{16}$ subsets of $\{0,1\}^4$.

In [ ]:
E4 = [(u, u ^ (1 << i)) for u in range(16) for i in range(4) if u < u ^ (1 << i)]
worst = 0
for S in range(1, 1 << 16):
    s = bin(S).count("1"); e = sum(1 for u, v in E4 if (S >> u) & 1 and (S >> v) & 1)
    worst = max(worst, e - s * log2(s) / 2)
print(round(worst, 9))

**Problem 8.15 (Erdős–Selfridge).** The criterion for clique games, and the largest $N$ it certifies for each $q$.

In [ ]:
for q in (3, 4, 5, 6, 8, 10, 12, 16, 20):
    N = q
    while comb(N + 1, q) * 2.0**(-comb(q, 2)) < 0.5: N += 1
    print(q, N, round(2**(q/2), 1), end=";  ")

**Problem 8.16 (Shearer, Loomis–Whitney).** Random finite sets in $\mathbb Z^3$, and triangles against $(2m)^{3/2}/6$.

In [ ]:
ok = True
for _ in range(200):
    S = {tuple(rng.integers(0, 5, 3)) for _ in range(random.randint(1, 60))}
    xy, yz, xz = ({(x, y) for x, y, z in S}, {(y, z) for x, y, z in S}, {(x, z) for x, y, z in S})
    ok &= len(S)**2 <= len(xy) * len(yz) * len(xz)
cube = {(x, y, z) for x in range(4) for y in range(4) for z in range(4)}
print(ok, len(cube)**2 == 16**3)
for n in (6, 20, 80):
    A = np.ones((n, n), dtype=int) - np.eye(n, dtype=int); m = A.sum() // 2; print(triangles(A), round((2*m)**1.5 / 6))

**Problem 8.17 (Chebyshev).** The moments of $\varphi(n)/n$: Euler products $M_1=\prod(1-p^{-2})$, $M_2=\prod(1-(2p-1)p^{-3})$, and
empirical averages up to $10^6$.

In [ ]:
N = 10**6
ph = np.arange(N + 1, dtype=np.float64)
for p in primerange(2, N + 1): ph[p::p] *= (1 - 1/p)
r = ph[1:] / np.arange(1, N + 1)
M = [float(np.mean(r**k)) for k in range(1, 5)]
M1 = np.prod([1 - p**-2.0 for p in primerange(2, 10**6)]); M2 = np.prod([1 - (2*p - 1) / p**3 for p in primerange(2, 10**6)])
print([round(x, 4) for x in M], round(M1, 4), round(M2, 4))
print(M[1] >= M[0]**2, M[2] >= M[0]*M[1], M[3] >= M[1]**2, M[3] >= M[0]*M[2])

**Problem 8.18 (Newton).** Stirling numbers of the first kind satisfy Newton's strong form of log-concavity.

In [ ]:
def stirling1(n):
    row = [1]
    for j in range(n):                       # multiply by (x + j)
        row = [(row[k-1] if k >= 1 else 0) + j * (row[k] if k < len(row) else 0) for k in range(len(row) + 1)]
    return row                               # row[k] = c(n, k)
for n in (5, 10, 30):
    c = stirling1(n)
    print(n, all(c[k]**2 * k * (n - k) >= c[k-1] * c[k+1] * (k+1) * (n-k+1) for k in range(1, n)), c.index(max(c)))

**Problem 8.19 (Bonferroni).** Derangements against the truncated series, error at most $1/(t+1)!$.

In [ ]:
def D(n): return round(factorial(n) / math.e) if n else 1
n = 10
for t in range(0, n + 1):
    s = sum(Fraction((-1)**j, factorial(j)) for j in range(t + 1)); err = abs(Fraction(D(n), factorial(n)) - s)
    print(t, err <= Fraction(1, factorial(t + 1)), (Fraction(D(n), factorial(n)) - s > 0) == (t % 2 == 1) or err == 0, end="; ")

**Problem 8.20 (primitive sets).** $\sum 1/(a\log a)$ over the primes ($1.6366\ldots$) and over the numbers with exactly two
prime factors, both primitive sets; the partial sums converge very slowly.

In [ ]:
X = 10**6
Om = np.zeros(X + 1, dtype=np.int8)
for p in primerange(2, X + 1):
    q = p
    while q <= X: Om[q::q] += 1; q *= p
a = np.arange(2, X + 1); w = 1 / (a * np.log(a))
print(round(w[Om[2:] == 1].sum(), 4), round(w[Om[2:] == 2].sum(), 4), round(w[Om[2:] == 3].sum(), 4))

The primes alone have $\sum_p\frac1{p\log p}=1.6366\ldots$; the partial sum up to $10^6$ is only $1.564$, since the tail beyond $x$ is about $\frac1{\log x}$, and the semiprimes
converge even more slowly. Erdős conjectured in 1988 that no primitive set beats the primes, and Jared Duker Lichtman proved it in 2022.

**Problem 8.21 (Ruzsa).** The triangle inequality and the covering lemma on random sets of integers.

In [ ]:
def S(X, Y, sign=1): return {x + sign*y for x in X for y in Y}
ok = True
for _ in range(300):
    A_, B_, C_ = (set(random.sample(range(60), random.randint(2, 15))) for _ in range(3))
    ok &= len(A_) * len(S(B_, C_, -1)) <= len(S(A_, B_, -1)) * len(S(A_, C_, -1))
print(ok)
A_ = set(range(0, 40, 2)) | {101}; B_ = set(range(0, 200, 7)); K = len(S(A_, B_)) / len(A_)
X, cov = [], set()                           # greedy maximal set X with A + x disjoint: then B is in X + A - A
for b in B_:
    if all(not (S(A_, {b}) & S(A_, {x})) for x in X): X.append(b)
AA = S(A_, A_, -1)
print(round(K, 2), len(X), all(any(b - x in AA for x in X) for b in B_))

**Problem 8.22 (champions of divisors).** Records of $d(n)/n^\alpha$ against the product formula and the rule of part (b), for $\alpha=\frac12,\frac13$ (sieve up to $10^6$) and $\alpha=\frac14$ (the predicted champion $21\,621\,600$).

In [ ]:
N = 10**6
d = np.zeros(N + 1, dtype=np.int32)
for k in range(1, N + 1): d[k::k] += 1
n = np.arange(N + 1, dtype=float); n[0] = 1
for al in (1/2, 1/3):
    r = d / n**al; k = int(np.argmax(r))
    prod = 1.0
    for p in primerange(2, 200):
        prod *= max((e + 1) / p**(e*al) for e in range(40))
    print(round(al, 3), k, round(r[k], 4), round(prod, 4))
def champion(al, P=50):
    n = 1
    for p in primerange(2, P):
        e = 0
        while (e + 2) / (e + 1) > p**al: e += 1
        n *= p**e
    return n
c4 = champion(1/4); print(champion(1/2), champion(1/3), c4, divisor_count(c4), round(divisor_count(c4) / c4**0.25, 4))

## Part B: laboratory

**L8.1 (distances in grids).** Count the distinct distances of the $k\times k$ grid for $k\le 60$ and compare with $\frac{n}{\sqrt{\log n}}$,
$n=k^2$. Which constant appears? (Landau: the number of sums of two squares up to $x$ is about $0.7642\,x/\sqrt{\log x}$.)

**L8.2 (sums or products).** Compute $\max(|A+A|,|A\cdot A|)$ for intervals, geometric progressions and sets of smooth numbers of
the same size $N$, and estimate the exponent $\log\max/\log N$. Which set comes closest to $N^{1}$?

**L8.3 (playing Erdős–Selfridge).** Program Breaker's potential strategy for the clique game on $K_N$ with $q=4$ and $N=7$
(where the criterion holds) and play it against a greedy Maker.

**L8.4 (the multiplication table).** Count the distinct entries $M(N)$ of the $N\times N$ table for $N\le2000$. Compare $M(N)$ with
the lower bound of Problem 8.5 and with Ford's order of magnitude $N^2/\bigl((\log N)^{\delta}(\log\log N)^{3/2}\bigr)$, $\delta=0.0860\ldots$

## Solutions

### L8.1

In [ ]:
res = []
for k in (10, 20, 30, 40, 60):
    vals = {i*i + j*j for i in range(k) for j in range(i, k)} - {0}
    n = k*k; res.append((k, len(vals), round(len(vals) / (n / sqrt(log(n))), 3)))
print(res)

The ratio creeps up very slowly, near $1.1$: the distances of the grid are the numbers $i^2+j^2$ with $i,j<k$, which are all the
sums of two squares below $k^2$ (about $0.76\,k^2/\sqrt{\log k^2}$ of them by Landau's theorem) plus part of those between $k^2$ and $2k^2$. The grid beats every
$n^{1-\varepsilon}$, which is why Erdős conjectured it is optimal, and Guth and Katz proved that no set does better than $cn/\log n$.

### L8.2

In [ ]:
def smooth(N, P=(2, 3, 5)):
    out, x = [], 1
    while len(out) < N:
        if all(p in P for p in factorint(x)): out.append(x)
        x += 1
    return out
for N in (20, 50, 100, 200):
    row = []
    for A in (list(range(1, N + 1)), [2**i for i in range(N)], smooth(N)):
        s, p = sp(A); row.append(round(log(max(s, p)) / log(N), 3))
    print(N, row)

Intervals have few sums but about $N^2/(\log N)^{c}$ products; geometric progressions have few products but all $\binom{N+1}2$
sums. Smooth numbers try to be good at both, and they keep the exponent lowest, but it still drifts above $1$. Erdős and
Szemerédi conjectured that $\max(|A+A|,|A\cdot A|)\ge N^{2-\varepsilon}$; the best known exponent is $\frac43+\frac2{1167}$ (Rudnev and
Stevens, 2020), not far beyond Elekes's $\frac54$ and Solymosi's $\frac43$.

### L8.3

In [ ]:
def play(N, q, maker):
    V = list(itertools.combinations(range(N), 2))
    H = [frozenset(itertools.combinations(c, 2)) for c in itertools.combinations(range(N), q)]
    mk, bk = set(), set()
    pot = lambda: sum(2.0**-len(h - mk) for h in H if not (h & bk))
    for turn in range(len(V)):
        free = [e for e in V if e not in mk and e not in bk]
        if not free: break
        if turn % 2 == 0: mk.add(maker(free, H, mk, bk))
        else:                                  # Breaker takes the free edge of largest weight
            bk.add(max(free, key=lambda e: sum(2.0**-len(h - mk) for h in H if e in h and not (h & bk))))
        if any(h <= mk for h in H): return "Maker", turn
    return "Breaker", round(pot(), 3)
greedy = lambda free, H, mk, bk: max(free, key=lambda e: sum(2.0**-len(h - mk) for h in H if e in h and not (h & bk)))
rand = lambda free, H, mk, bk: random.choice(free)
print(round(comb(7, 4) * 2.0**-6, 3), play(7, 4, greedy), play(7, 4, rand))
print(round(comb(12, 4) * 2.0**-6, 3), play(12, 4, greedy))

For $N=7$, $\binom74 2^{-6}=0.547$ is just above $\frac12$, yet Breaker still wins: the criterion is sufficient, not necessary, and
after Maker's first move the potential is what matters. For $N=12$ the potential starts far above $\frac12$ and greedy Maker
wins quickly. Maker surely wins when $N\ge4^q$: some colour class then contains a $K_q$ by Ramsey's theorem (Erdős–Szekeres bound),
and a strategy-stealing argument shows that the first player cannot lose such a game. Beck showed that the true threshold is
$N=2^{q/2+o(q)}$, so the Erdős–Selfridge criterion is sharp to first order.

### L8.4

In [ ]:
Ns = [100, 200, 500, 1000, 2000]; out = []
dlt = 1 - (1 + log(log(2))) / log(2)
for N in Ns:
    seen = np.zeros(N*N + 1, dtype=bool)
    for a in range(1, N + 1): seen[a * np.arange(a, N + 1)] = True
    M_ = int(seen.sum()); E = Ex_fast(N)
    out.append((N, M_, round(M_ / (N**4 / E), 2), round(M_ / (N*N / (log(N)**dlt * log(log(N))**1.5)), 3)))
print(round(dlt, 4)); print(out)

The energy bound $N^4/E_\times\approx\frac{\pi^2}{12}\frac{N^2}{\log N}$ is too small by a factor that keeps growing, while the ratio to
Ford's order grows much more slowly (its limit, if it exists, is unknown; at this range the lower-order terms still matter). Erdős (1955, 1960) showed that $M(N)=o(N^2)$ with the exponent $\delta$; Ford (2008) found the true order.
The energy method gives the right order for sums or products of structured sets, but here a positive proportion of the
numbers up to $N^2$ with a typical number of prime factors is missed, and that is what the factor $(\log N)^{\delta}$ counts.